In [1]:
!pip install lightgbm

In [2]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import seaborn as sns

---
## <u>Load Dataset</u>

In [3]:
data = sns.load_dataset("diamonds")
data.info() # no missing values
data["color"].nunique() # 7 categories
data["clarity"].nunique() # 8 categories

data["cut"].nunique() # 5 categories, we choose this as our labels (y)

data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   price    53940 non-null  int64   
 7   x        53940 non-null  float64 
 8   y        53940 non-null  float64 
 9   z        53940 non-null  float64 
dtypes: category(3), float64(6), int64(1)
memory usage: 3.0 MB


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


---
## <u>Train Test Split</u>

In [4]:
X = data.drop(columns = ["cut"])
y = data["cut"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

---
## <u>Feature Encoding</u>

In [5]:
# 1. get categorical and numerical column names
num_cols = X_train.select_dtypes(include = ["number"]).columns
cat_cols = X_train.select_dtypes(include = ["category"]).columns

# 2. make preprocessor
preprocessor = ColumnTransformer(
    transformers = [
        ("ohe", OneHotEncoder(sparse_output = False, handle_unknown = "ignore"), cat_cols),
        ("scaler", StandardScaler(), num_cols)
    ]
)

# 3. set output to df
preprocessor.set_output(transform="pandas")

# 4. use the preprocessor on X (features)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# 5. use label encoder on y (labels)
le = LabelEncoder()
y_train_processed = le.fit_transform(y_train)
y_test_processed = le.transform(y_test)

---
## <u>Create, Train and Predict</u>

In [6]:
# We set verbose = -1 inside LGBMClassifier because LightGBM uses -1 to mean "Fatal errors only / completely silent"
# If we used 0 here, LightGBM would still print warnings and minor logs for all model fits.
lightGBM_classifier = lgb.LGBMClassifier(random_state = 42, verbose = -1)
lightGBM_classifier.fit(X_train, y_train)

y_train_pred = lightGBM_classifier.predict(X_train)
y_test_pred = lightGBM_classifier.predict(X_test)

---
## <u>Evaluate</u>

In [7]:
print("For LightGBM Classifier (baseline) :-\n")

print("Training scores :-")
print("Train Accuracy : ", accuracy_score(y_train, y_train_pred))
print("Train classification report  : \n", classification_report(y_train, y_train_pred))


print("\nTesting scores :-")
print("Test Accuracy : ", accuracy_score(y_test, y_test_pred))
print("Test classification report  : \n", classification_report(y_test, y_test_pred))


For LightGBM Classifier (baseline) :-

Training scores :-
Train Accuracy :  0.8295096403411197
Train classification report  : 
               precision    recall  f1-score   support

        Fair       0.98      0.97      0.97      1275
        Good       0.88      0.76      0.82      3902
       Ideal       0.84      0.94      0.88     17259
     Premium       0.85      0.83      0.84     11016
   Very Good       0.75      0.65      0.70      9700

    accuracy                           0.83     43152
   macro avg       0.86      0.83      0.84     43152
weighted avg       0.83      0.83      0.83     43152


Testing scores :-
Test Accuracy :  0.7994067482387839
Test classification report  : 
               precision    recall  f1-score   support

        Fair       0.92      0.90      0.91       335
        Good       0.80      0.69      0.74      1004
       Ideal       0.82      0.92      0.87      4292
     Premium       0.83      0.81      0.82      2775
   Very Good       0.69  

---
## <u>Hyperparameter Tuning</u>

In [12]:
# 1. Initialise the steps and make the pipeline
# Set n_jobs=1 inside LGBMClassifier so it doesn't fight Scikit-Learn for CPU cores

steps = [("lgbmc", LGBMClassifier(random_state = 42, verbose = -1, n_jobs = 1))] 
pipeline = Pipeline(steps)

# 2. define the paramter grid 

param_grid = {
    "lgbmc__max_depth" : [4, 6, 8],
    "lgbmc__learning_rate" : [0.2 ,0.1, 0.01], 
    "lgbmc__n_estimators" : [100, 200, 300],
    "lgbmc__subsample" : [0.6, 0.8, 1.0],
    "lgbmc__reg_alpha" : [0.1, 1, 3], # L1 regularization term on weights.
    "lgbmc__reg_lambda" : [0.1, 1, 3], # L2 regularization term on weights.
    "lgbmc__min_child_samples" : [50, 100, 150], #  Minimum number of data needed in a child (leaf)
    "lgbmc__num_leaves" : [21, 31, 41] # Maximum tree leaves for base learners
}

# 3. cross validation blueprint

lightGBM_classifier_cv = RandomizedSearchCV(
    pipeline,
    param_grid,
    cv = 3,
    n_iter = 20, # Just randomly pick 20 combinations out of the 2,187 and test only those.
    scoring="accuracy", 
    n_jobs = 1,
    verbose=0 # We set verbose = 0 inside GridSearchCV because Scikit-Learn uses 0 to mean "completely silent".
)

# 4. make and train the model

lightGBM_classifier_cv.fit(X_train, y_train)

# 5. make predictions

y_train_pred = lightGBM_classifier_cv.predict(X_train)
y_test_pred = lightGBM_classifier_cv.predict(X_test)

# 6. evaluate 

print("For LightGBM Classifier (hyperparameter tuning) :-\n")

print("Best parameters found: ", lightGBM_classifier_cv.best_params_)

print("\nTraining scores :-")
print("Train Accuracy : ", accuracy_score(y_train, y_train_pred))
print("Train classification report  : \n", classification_report(y_train, y_train_pred))

print("\nTesting scores :-")
print("Test Accuracy : ", accuracy_score(y_test, y_test_pred))
print("Test classification report  : \n", classification_report(y_test, y_test_pred))

For LightGBM Classifier (hyperparameter tuning) :-

Best parameters found:  {'lgbmc__subsample': 0.8, 'lgbmc__reg_lambda': 1, 'lgbmc__reg_alpha': 1, 'lgbmc__num_leaves': 41, 'lgbmc__n_estimators': 100, 'lgbmc__min_child_samples': 100, 'lgbmc__max_depth': 8, 'lgbmc__learning_rate': 0.2}

Training scores :-
Train Accuracy :  0.8472840192806822
Train classification report  : 
               precision    recall  f1-score   support

        Fair       0.98      0.95      0.97      1275
        Good       0.90      0.81      0.85      3902
       Ideal       0.85      0.94      0.89     17259
     Premium       0.86      0.85      0.85     11016
   Very Good       0.78      0.69      0.73      9700

    accuracy                           0.85     43152
   macro avg       0.87      0.85      0.86     43152
weighted avg       0.85      0.85      0.84     43152


Testing scores :-
Test Accuracy :  0.8029291805710048
Test classification report  : 
               precision    recall  f1-score   s